In [1]:
from pyscf import gto, scf, dft, lib
import numpy as np
np.set_printoptions(10, linewidth=150)

## Setups

In [2]:
mol = gto.Mole(atom="C; H 1 0.94; H 1 0.94 2 104.5", spin=2, charge=2, basis="def2-TZVP").build()

In [3]:
mf = scf.UKS(mol, xc="TPSS0").run()

converged SCF energy = -37.7173135264959  <S^2> = 2.0026514  2S+1 = 3.0017671


In [4]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [5]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [6]:
grids = mf.grids

In [7]:
ni = dft.numint.NumInt()

In [8]:
np.savez("ch2", **arrays)

## Get rho by heterogeneous brakets

In [9]:
# fake a heterogeneous brakets by forming its symmetric dm
dms = mf.mo_coeff[:, :, :4] @ mf.mo_coeff[0, :, :4].T

In [10]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = np.array([ni.eval_rho(mol, ao[0], dms[s], xctype="lda") for s in (0, 1)])
rho_sigma = np.array([ni.eval_rho(mol, ao[0:4], dms[s], xctype="gga") for s in (0, 1)])
rho_tau = np.array([ni.eval_rho(mol, ao[0:4], dms[s], xctype="mgga", with_lapl=False) for s in (0, 1)])
rho_lapl = np.array([ni.eval_rho(mol, ao[0:10], dms[s], xctype="mgga", with_lapl=True) for s in (0, 1)])

In [11]:
rho_rho.shape, lib.fp(rho_rho)

((2, 33736), np.float64(90.19801284927166))

In [12]:
rho_sigma.shape, lib.fp(rho_sigma)

((2, 4, 33736), np.float64(358.2987825175244))

In [13]:
rho_tau.shape, lib.fp(rho_tau)

((2, 5, 33736), np.float64(5531.545589708348))

In [14]:
rho_lapl_ = rho_lapl[:, [0, 1, 2, 3, 5, 4], :]
rho_lapl_.shape, lib.fp(rho_lapl_[:, :5, :]), lib.fp(rho_lapl_[:, 5, :])

((2, 6, 33736), np.float64(5531.545589708348), np.float64(-795547.4132924974))

## Get rho by dm

In [15]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = np.array([ni.eval_rho(mol, ao[0], rdm1[s], xctype="lda") for s in (0, 1)])
rho_sigma = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="gga") for s in (0, 1)])
rho_tau = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="mgga", with_lapl=False) for s in (0, 1)])
rho_lapl = np.array([ni.eval_rho(mol, ao[0:10], rdm1[s], xctype="mgga", with_lapl=True) for s in (0, 1)])

In [16]:
rho_rho.shape, lib.fp(rho_rho)

((2, 33736), np.float64(90.16002674073957))

In [17]:
rho_sigma.shape, lib.fp(rho_sigma)

((2, 4, 33736), np.float64(369.81784285462714))

In [18]:
rho_tau.shape, lib.fp(rho_tau)

((2, 5, 33736), np.float64(5587.859016487352))

In [19]:
rho_lapl_ = rho_lapl[:, [0, 1, 2, 3, 5, 4], :]
rho_lapl_.shape, lib.fp(rho_lapl_[:, :5, :]), lib.fp(rho_lapl_[:, 5, :])

((2, 6, 33736), np.float64(5587.859016487352), np.float64(-783527.536833374))

## xc_eff

### RHO (LDA)

In [20]:
result = ni.eval_xc_eff("LDA_X", rho_rho, deriv=3, spin=1)

In [21]:
[r.shape for r in result]

[(33736,), (2, 1, 33736), (2, 1, 2, 1, 33736), (2, 1, 2, 1, 2, 1, 33736)]

In [22]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_rho[:, None, :] * weights),
    lib.fp(result[3] * rho_rho[:, None, :] * rho_rho[:, None, None, None, :] * weights),
])
fps

array([-0.0050679843,  0.1013037713, -0.0417593614,  0.0281257118])

### SIGMA (GGA)

In [23]:
result = ni.eval_xc_eff("GGA_X_PBE", rho_sigma, deriv=3, spin=1)

In [24]:
[r.shape for r in result]

[(33736,), (2, 4, 33736), (2, 4, 2, 4, 33736), (2, 4, 2, 4, 2, 4, 33736)]

In [25]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_sigma * weights),
    lib.fp(result[3] * rho_sigma * rho_sigma[:, :, None, None, :] * weights),
])
fps

array([ 0.0174826167, -0.0688243866, -0.0998561381,  0.1192110757])

### TAU (MGGA)

In [26]:
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3, spin=1)

In [27]:
[r.shape for r in result]

[(33736,), (2, 5, 33736), (2, 5, 2, 5, 33736), (2, 5, 2, 5, 2, 5, 33736)]

In [28]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_tau * weights),
    lib.fp(result[3] * rho_tau * rho_tau[:, :, None, None, :] * weights),
])
fps

array([ 0.0070440179,  0.0931735851, -0.0147352726,  1.3842052628])

In [29]:
%%time
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3, spin=1)

CPU times: user 3.03 s, sys: 246 ms, total: 3.27 s
Wall time: 160 ms


## exc, vxc, fxc, kxc

For spin-polarized (UKS) systems, the key differences from RKS are:
- The density matrix has separate alpha/beta components: `dm0` shape `(2, nao, nao)`
- The perturbed DMs are spin-separated: `dm1` shape `(2, ndm, nao, nao)`
- `vxc_eff` has shape `(2, nvar, ngrids)`: effective potential for each spin channel
- `fxc_eff` has shape `(2, nvar, 2, nvar, ngrids)`: includes cross-spin coupling (e.g., $f^{\alpha\beta}_{xc}$)
- The vxc AO matrix must be constructed separately for each spin
- The fxc contraction requires `einsum` over both spin channels

In [49]:
# Setting up (density)
dm0 = rdm1
dm1 = (mol.intor('int1e_r') + mol.intor('int1e_giao_irjxp'))[None, :, :, :] * rdm1[:, None, :, :]
dm2 = mol.intor('int1e_rr')[None, :, :, :] * rdm1[:, None, :, :]

### RHO (LDA)

In [65]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = np.array([ni.eval_rho(mol, ao[0], dm0[s], xctype='lda') for s in (0, 1)])
rho1 = np.array([[ni.eval_rho(mol, ao[0], dm1[s, x], xctype='lda') for s in (0, 1)] for x in range(dm1.shape[1])])
rho2 = np.array([[ni.eval_rho(mol, ao[0], dm2[s, y], xctype='lda') for s in (0, 1)] for y in range(dm2.shape[1])])
rho1 = rho1.reshape([dm1.shape[1], 2, -1, ngrids])
rho2 = rho2.reshape([dm2.shape[1], 2, -1, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff('LDA_X', rho0, deriv=3, spin=1)

In [66]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_uks(mol, grids, 'LDA_X', dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -4.7040426008, -12.7427734694])

In [67]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * (rho0[0] + rho0[1]) * weights).sum()
vxc_recap = np.array([ao[0].T * (weights * vxc_eff[s, 0]) @ ao[0] for s in range(2)])
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [68]:
# canonical way of fxc
fxc = ni.nr_uks_fxc(mol, grids, 'LDA_X', dm0, dm1)
fxc.shape, lib.fp(fxc)

((2, 3, 43, 43), np.float64(-0.3142431089581791))

In [ ]:
# naive blas-3 way of fxc
# fxc_eff: (2, nvar, 2, nvar, ngrids) — cross-spin coupling included
# rho1: (2, ndm, nvar, ngrids)
# Contract over s2, v2 to get fxc_contract: (s1, i, v1, g)
fxc_contract = np.einsum('abcdg,cidg->aibg', fxc_eff, rho1) * weights
fxc_recap = np.zeros([2, dm1.shape[1], nao, nao])
for s in range(2):
    for x in range(dm1.shape[1]):
        fxc_recap[s, x] = ao[0].T * fxc_contract[s, x, 0] @ ao[0]
assert np.allclose(fxc_recap, fxc)

In [55]:
# naive blas-3 way of kxc
# kxc_eff: (2, nvar, 2, nvar, 2, nvar, ngrids) — cross-spin coupling
# rho1: (2, ndm, nvar, ngrids), rho2: (2, ndm2, nvar, ngrids)
# Contract over s2,v2 and s3,v3 to get kxc_contract: (s1, j, i, v1, g)
kxc_contract = np.einsum('abcdefg,cidg,ejfg->ajibg', kxc_eff, rho1, rho2) * weights
kxc_recap = np.zeros([2, dm2.shape[1], dm1.shape[1], nao, nao])
for s in range(2):
    for y in range(dm2.shape[1]):
        for x in range(dm1.shape[1]):
            kxc_recap[s, y, x] = ao[0].T * kxc_contract[s, y, x, 0] @ ao[0]
kxc_recap.shape, lib.fp(kxc_recap)

((2, 9, 3, 43, 43), np.float64(0.21014061808695567))

### SIGMA (GGA)

In [69]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = np.array([ni.eval_rho(mol, ao[:4], dm0[s], xctype='gga') for s in (0, 1)])
rho1 = np.array([[ni.eval_rho(mol, ao[:4], dm1[s, x], xctype='gga') for s in (0, 1)] for x in range(dm1.shape[1])])
rho2 = np.array([[ni.eval_rho(mol, ao[:4], dm2[s, y], xctype='gga') for s in (0, 1)] for y in range(dm2.shape[1])])
nvar = rho0.shape[1]
rho1 = rho1.reshape([2, dm1.shape[1], nvar, ngrids])
rho2 = rho2.reshape([2, dm2.shape[1], nvar, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff('GGA_X_PBE', rho0, deriv=3, spin=1)

In [70]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_uks(mol, grids, 'GGA_X_PBE', dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -5.2725625947, -13.134537099 ])

In [71]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * (rho0[0, 0] + rho0[1, 0]) * weights).sum()
vxc_recap = np.zeros([2, nao, nao])
for s in range(2):
    vxc_recap[s] = (
        + 0.5 * ao[0].T * (weights * vxc_eff[s, 0]) @ ao[0]
        + ao[1].T * (weights * vxc_eff[s, 1]) @ ao[0]
        + ao[2].T * (weights * vxc_eff[s, 2]) @ ao[0]
        + ao[3].T * (weights * vxc_eff[s, 3]) @ ao[0]
    )
    vxc_recap[s] += vxc_recap[s].swapaxes(-1, -2)
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [72]:
# canonical way of fxc
fxc = ni.nr_uks_fxc(mol, grids, 'GGA_X_PBE', dm0, dm1)
fxc.shape, lib.fp(fxc)

((2, 3, 43, 43), np.float64(-0.33975230052462946))

In [86]:
# naive blas-3 way of fxc
fxc_eff_reshape = fxc_eff.reshape(2, nvar, 2 * nvar, ngrids)  # [   2, v0, 2, v1, g] -> [   2, v0, d1=2*v1, g]
rho1_reshape = rho1.reshape(dm1.shape[1], 2 * nvar, ngrids)   # [x,        2, v1, g] -> [x,        d1=2*v1, g]
fxc_contract = (fxc_eff_reshape[None, :, :, :, :] * rho1_reshape[:, None, None, :, :]).sum(axis=-2) * weights
#                                [x]  2 v0 d1  g                 x   [2]  [v0] d1  g
# shape : [len(dm1), 2, nvar, ngrids]
fxc_recap = np.zeros([dm1.shape[1], 2, nao, nao])
for x in range(dm1.shape[1]):
    for s in range(2):
        fxc_recap[x, s] = (
            + 0.5 * ao[0].T * fxc_contract[x, s, 0] @ ao[0]
            + ao[1].T * fxc_contract[x, s, 1] @ ao[0]
            + ao[2].T * fxc_contract[x, s, 2] @ ao[0]
            + ao[3].T * fxc_contract[x, s, 3] @ ao[0]
        )
fxc_recap += fxc_recap.swapaxes(-1, -2)
# we may have different convention for the order spin (s) and derivative (x) to PySCF
assert np.allclose(fxc_recap.transpose(1, 0, 3, 2), fxc)
# shape, pyscf convention, rest convention
fxc_recap.shape, lib.fp(fxc_recap.transpose(1, 0, 3, 2)), lib.fp(fxc_recap)

((3, 2, 43, 43),
 np.float64(-0.33975230052462935),
 np.float64(-0.1379220511462902))

In [ ]:
# naive blas-3 way of kxc
kxc_eff_reshape = kxc_eff.reshape(2, nvar, 2 * nvar, 2 * nvar, ngrids)  # [      2, v0, 2, v2, 2, v1, g] -> [      2, v0, d2=2*v2, d1=2*v1, g]
rho1_reshape = rho1.reshape(dm1.shape[1], 2 * nvar, ngrids)             # [   x,               2, v1, g] -> [   x,                 d1=2*v1, g]
rho2_reshape = rho2.reshape(dm2.shape[1], 2 * nvar, ngrids)             # [y,           2, v2,        g] -> [y,           d2=2*v2,          g]
kxc_contract = (
    kxc_eff_reshape[None, None, :, :, :, :, :]  # [y], [x], 2, v0, d2, d1, g
    * rho1_reshape[None, :, None, None, None, :, :]  # [y], x, [2], [v0], [d2], d1, g
    * rho2_reshape[:, None, None, None, :, None, :]  # y, [x], [2], [v0], d2, [d1], g
).sum(axis=(-2, -3)) * weights
# shape : [len(dm2), len(dm1), 2, nvar, ngrids]
kxc_recap = np.zeros([dm2.shape[1], dm1.shape[1], 2, nao, nao])
for y in range(dm2.shape[1]):
    for x in range(dm1.shape[1]):
        for s in range(2):
            kxc_recap[y, x, s] = (
                + 0.5 * ao[0].T * kxc_contract[y, x, s, 0] @ ao[0]
                + ao[1].T * kxc_contract[y, x, s, 1] @ ao[0]
                + ao[2].T * kxc_contract[y, x, s, 2] @ ao[0]
                + ao[3].T * kxc_contract[y, x, s, 3] @ ao[0]
            )
kxc_recap += kxc_recap.swapaxes(-1, -2)
# shape, pyscf convention, rest convention
kxc_recap.shape, lib.fp(kxc_recap.transpose(2, 0, 1, 3, 4)), lib.fp(kxc_recap)

((9, 3, 2, 43, 43),
 np.float64(0.21380601801163712),
 np.float64(0.2770362015666324))

In [81]:
kxc_contract[y, x, s, 0].shape

(8, 8, 33736)

In [ ]:
((2, 9, 3, 43, 43), np.float64(0.21380601801163715))

### TAU (MGGA)

In [43]:
# Setting up (density grids and eff potentials)
(_, ngrids, nao) = ao.shape
rho0 = np.array([ni.eval_rho(mol, ao[:4], dm0[s], xctype='mgga', with_lapl=False) for s in (0, 1)])
rho1 = np.array([[ni.eval_rho(mol, ao[:4], dm1[s, x], xctype='mgga', with_lapl=False) for x in range(dm1.shape[1])] for s in (0, 1)])
rho2 = np.array([[ni.eval_rho(mol, ao[:4], dm2[s, y], xctype='mgga', with_lapl=False) for y in range(dm2.shape[1])] for s in (0, 1)])
nvar = rho0.shape[1]
rho1 = rho1.reshape([2, dm1.shape[1], nvar, ngrids])
rho2 = rho2.reshape([2, dm2.shape[1], nvar, ngrids])
exc_eff, vxc_eff, fxc_eff, kxc_eff = ni.eval_xc_eff('HYB_MGGA_XC_TPSSH', rho0, deriv=3, spin=1)

In [44]:
# canonical way of exc/vxc
nelec, exc, vxc = ni.nr_uks(mol, grids, 'HYB_MGGA_XC_TPSSH', dm0)
np.array([
    exc,
    lib.fp(vxc)
])

array([ -4.9638946892, -12.384391087 ])

In [45]:
# naive blas-3 way of exc/vxc
exc_recap = (exc_eff * (rho0[0, 0] + rho0[1, 0]) * weights).sum()
vxc_recap = np.zeros([2, nao, nao])
for s in range(2):
    vxc_recap[s] = (
        + 0.5 * ao[0].T * (weights * vxc_eff[s, 0]) @ ao[0]
        + ao[1].T * (weights * vxc_eff[s, 1]) @ ao[0]
        + ao[2].T * (weights * vxc_eff[s, 2]) @ ao[0]
        + ao[3].T * (weights * vxc_eff[s, 3]) @ ao[0]
        + 0.25 * ao[1].T * (weights * vxc_eff[s, 4]) @ ao[1]
        + 0.25 * ao[2].T * (weights * vxc_eff[s, 4]) @ ao[2]
        + 0.25 * ao[3].T * (weights * vxc_eff[s, 4]) @ ao[3]
    )
    vxc_recap[s] += vxc_recap[s].swapaxes(-1, -2)
assert np.allclose(exc_recap, exc)
assert np.allclose(vxc_recap, vxc)

In [46]:
# canonical way of fxc
fxc = ni.nr_uks_fxc(mol, grids, 'HYB_MGGA_XC_TPSSH', dm0, dm1)
fxc.shape, lib.fp(fxc)

((2, 3, 43, 43), np.float64(36.975889233368875))

In [47]:
# naive blas-3 way of fxc
fxc_contract = np.einsum('abcdg,cidg->aibg', fxc_eff, rho1) * weights
#                        [s1] v1 [s2] v2  g    [s2] i  v2  g   → [s1] i  v1  g
# shape : [2, len(dm1), nvar, ngrids]
fxc_recap = np.zeros([2, dm1.shape[1], nao, nao])
for s in range(2):
    for x in range(dm1.shape[1]):
        fxc_recap[s, x] = (
            + 0.5 * ao[0].T * fxc_contract[s, x, 0] @ ao[0]
            + ao[1].T * fxc_contract[s, x, 1] @ ao[0]
            + ao[2].T * fxc_contract[s, x, 2] @ ao[0]
            + ao[3].T * fxc_contract[s, x, 3] @ ao[0]
            + 0.25 * ao[1].T * fxc_contract[s, x, 4] @ ao[1]
            + 0.25 * ao[2].T * fxc_contract[s, x, 4] @ ao[2]
            + 0.25 * ao[3].T * fxc_contract[s, x, 4] @ ao[3]
        )
    fxc_recap[s] += fxc_recap[s].swapaxes(-1, -2)
assert np.allclose(fxc_recap, fxc)

In [48]:
# naive blas-3 way of kxc
kxc_contract = np.einsum('abcdefg,cidg,ejfg->ajibg', kxc_eff, rho1, rho2) * weights
#                        [s1] v1 [s2] v2 [s3] v3  g    [s2] ib v2  g   [s3] jc v3  g   → [s1] jc ib v1  g
# shape : [2, len(dm2), len(dm1), nvar, ngrids]
kxc_recap = np.zeros([2, dm2.shape[1], dm1.shape[1], nao, nao])
for s in range(2):
    for y in range(dm2.shape[1]):
        for x in range(dm1.shape[1]):
            kxc_recap[s, y, x] = (
                + 0.5 * ao[0].T * kxc_contract[s, y, x, 0] @ ao[0]
                + ao[1].T * kxc_contract[s, y, x, 1] @ ao[0]
                + ao[2].T * kxc_contract[s, y, x, 2] @ ao[0]
                + ao[3].T * kxc_contract[s, y, x, 3] @ ao[0]
                + 0.25 * ao[1].T * kxc_contract[s, y, x, 4] @ ao[1]
                + 0.25 * ao[2].T * kxc_contract[s, y, x, 4] @ ao[2]
                + 0.25 * ao[3].T * kxc_contract[s, y, x, 4] @ ao[3]
            )
    kxc_recap[s] += kxc_recap[s].swapaxes(-1, -2)
kxc_recap.shape, lib.fp(kxc_recap)

((2, 9, 3, 43, 43), np.float64(155.06237016483738))